# Transformación y Limpieza de los CSV de Origen (previo a la ETL local)

**Objetivo de este cuaderno:** este notebook no forma parte de la secuencia de modelado (`notebooks/01`-`04`), sino que es un paso preparatorio de la **ETL**. Varios CSV de origen de eICU-CRD (`pastHistory.csv`, `physicalExam.csv`, `diagnosis.csv`, `treatment.csv`, `admissionDx.csv`) tienen columnas que concatenan múltiples niveles de información con `/` o `|` (por ejemplo `pasthistorypath` o `treatmentstring`), que no son directamente insertables en el modelo relacional de la base de datos local. Aquí se transforman esos CSV originales en los CSV "procesados" que se guardan en `CSV_filtrados/`, separando cada columna concatenada en sus componentes, renombrando columnas al esquema de la BD y eliminando duplicados. Estos CSV procesados son los que después carga el proyecto ETL.


## 1. Configuración inicial
Importación de librerías y creación de la carpeta de salida (`CSV_filtrados/`) donde se guardarán todos los CSV ya procesados.

In [1]:
import os
import pandas as pd
import re

In [2]:
# 1. Configurar la ruta y crear la carpeta si no existe
carpeta_destino = '../data/CSV_filtrados'
if not os.path.exists(carpeta_destino):
    os.makedirs(carpeta_destino)
    print(f"Carpeta '{carpeta_destino}' creada.")

Carpeta '../data/CSV_filtrados' creada.


## 2. Tabla Patient
De `patient.csv` extraemos identificador de paciente, género y etnia. Nos quedamos con un único registro por paciente (`keep='first'`), ya que un mismo paciente (`UniquePID`) puede tener varios ingresos registrados en el dataset original.

In [3]:
# 1. Leer el archivo original
df = pd.read_csv('../data/csv_origen/patient.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df.head()

,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,...,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,141168,128919,Female,70,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",152.4,15:54:00,...,Direct Admit,1,admit,84.3,85.8,03:50:00,3596,Death,Expired,002-34851
1,141178,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,...,Emergency Department,1,admit,54.4,54.4,09:18:00,8,Step-Down Unit (SDU),Alive,002-33870
2,141179,128927,Female,52,Caucasian,60,83,NaN,162.6,08:56:00,...,ICU to SDU,2,stepdown/other,NaN,60.4,19:20:00,2042,Home,Alive,002-33870
3,141194,128941,Male,68,Caucasian,73,92,"Sepsis, renal/UTI (including bladder)",180.3,18:18:40,...,Floor,1,admit,73.9,76.7,15:31:00,4813,Floor,Alive,002-5276
4,141196,128943,Male,71,Caucasian,67,109,NaN,162.6,20:21:00,...,ICU to SDU,2,stepdown/other,NaN,63.2,22:23:00,1463,Floor,Alive,002-37665


In [4]:
# 3. Seleccionar y copiar las columnas necesarias
columnas_interes = ['uniquepid', 'gender', 'ethnicity']
df_reducido = df[columnas_interes].copy()

df_reducido.head()

,uniquepid,gender,ethnicity
0,002-34851,Female,Caucasian
1,002-33870,Female,Caucasian
2,002-33870,Female,Caucasian
3,002-5276,Male,Caucasian
4,002-37665,Male,Caucasian


In [5]:
# 4. Renombrar las columnas
df_reducido.columns = ['UniquePID', 'Gender', 'Ethnicity']
df_reducido.head()

,UniquePID,Gender,Ethnicity
0,002-34851,Female,Caucasian
1,002-33870,Female,Caucasian
2,002-33870,Female,Caucasian
3,002-5276,Male,Caucasian
4,002-37665,Male,Caucasian


In [6]:
# 5. Filtrado de duplicados para conseguir registros únicos
# El parámetro keep='first' asegura que si un paciente tiene 3 ingresos, nos quedamos con el primero
df_patient_unico = df_reducido.drop_duplicates(subset=['UniquePID'], keep='first').reset_index(drop=True)

In [7]:
# 6. Definir la ruta completa del archivo
ruta_final = os.path.join(carpeta_destino, 'Patient.csv')

# 7. Guardar con separador de punto y coma
df_patient_unico.to_csv(ruta_final, sep=';', index=False)

print(f"¡Éxito! El archivo se ha guardado en: {ruta_final}")

¡Éxito! El archivo se ha guardado en: ../data/CSV_filtrados\Patient.csv


## 3. Tabla PastHist_AdmFact

### 3.1. Análisis de Terminología y Abreviaturas con "/"
En este apartado, realizamos una exploración exhaustiva de todas las columnas del dataset original PastHist_AdmFact.csv con el objetivo de identificar términos técnicos, unidades de medida o abreviaturas médicas que utilizan el símbolo de la barra diagonal (/).

Objetivo: Identificar cadenas de caracteres específicas (como por ejemplo h/o para history of) que suelen ser críticas para la limpieza de datos y la normalización de texto en registros clínicos.

In [8]:
# Cargamos el archivo
df = pd.read_csv('../data/csv_origen/PastHist_AdmFact.csv', sep=';')

def extraer_terminos_con_slash(dataframe):
    encontrados = set()
    
    # Recorremos cada columna del dataframe
    for col in dataframe.columns:
        # Convertimos a string y eliminamos nulos para el procesamiento
        contenido = dataframe[col].astype(str)
        
        for texto in contenido:
            # Buscamos patrones que tengan caracteres (que no sean espacios) 
            # antes y después de la barra diagonal
            matches = re.findall(r'\S+/\S+', texto)
            for m in matches:
                # Limpiamos signos de puntuación comunes al final del término
                termino_limpio = m.strip('.,;()[]{}')
                encontrados.add(termino_limpio)
                
    return sorted(list(encontrados))

# Ejecución
resultados = extraer_terminos_con_slash(df)

print(f"Se han encontrado {len(resultados)} términos únicos:")
for item in resultados:
    print(f"- {item}")

Se han encontrado 4 términos únicos:
- A/V
- FEV1/FVC
- h/o
- s/p


### 3.2. Transformación de PastHistory
La columna `pasthistorypath` concatena varios niveles de información separados por `/`. Antes de dividirla, protegemos temporalmente las abreviaturas detectadas en el análisis anterior (`A/V`, `FEV1/FVC`, `h/o`, `s/p`) para que el `split` no las trocee por error, y las restauramos justo después de separar la columna.

In [10]:
# 1. Leer el archivo original
df_ph = pd.read_csv('../data/csv_origen/pastHistory.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df_ph.head()

,pasthistoryid,patientunitstayid,pasthistoryoffset,pasthistoryenteredoffset,pasthistorynotetype,pasthistorypath,pasthistoryvalue,pasthistoryvaluetext
0,1141827,141168,60,72,Comprehensive Progress,notes/Progress Notes/Past History/Organ System...,AS,AS
1,1234145,141168,114,118,Comprehensive Progress,notes/Progress Notes/Past History/Organ System...,renal failure- not currently dialyzed,renal failure- not currently dialyzed
2,1141831,141168,60,72,Comprehensive Progress,notes/Progress Notes/Past History/Organ System...,CHF - class II,CHF - class II
3,1234141,141168,114,118,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,Performed,Performed
4,1141830,141168,60,72,Comprehensive Progress,notes/Progress Notes/Past History/Organ System...,hypertension requiring treatment,hypertension requiring treatment


In [11]:
# 2. Definir excepciones y protegerlas
# Reemplazamos las barras en las cadenas específicas por un marcador temporal (ej. "TEMP_SLASH")
excepciones = ['A/V', 'FEV1/FVC', 'h/o', 's/p']
path_protegido = df_ph['pasthistorypath'].copy()

for exc in excepciones:
    path_protegido = path_protegido.str.replace(exc, exc.replace('/', 'TEMP_SLASH'), regex=False)

In [12]:
# 3. Seleccionar y copiar las columnas necesarias
columnas_base = ['patientunitstayid', 'pasthistoryoffset']
df_ph_final = df_ph[columnas_base].copy()
df_ph_final.head()

,patientunitstayid,pasthistoryoffset
0,141168,60
1,141168,114
2,141168,60
3,141168,114
4,141168,60


In [13]:
# Renombrar las columnas
df_ph_final.columns = ['Patient', 'Offset']
df_ph_final.head()

,Patient,Offset
0,141168,60
1,141168,114
2,141168,60
3,141168,114
4,141168,60


In [14]:
# 2. Dividir 'pasthistorypath'
# Usamos expand=True para obtener un DataFrame de columnas
split_data_ph = path_protegido.str.split('/', expand=True)

In [15]:
split_data_ph.head()

,0,1,2,3,4,5,6,7,8
0,notes,Progress Notes,Past History,Organ Systems,Cardiovascular (R),Valve disease,AS,None,None
1,notes,Progress Notes,Past History,Organ Systems,Renal (R),Renal Failure,renal failure- not currently dialyzed,None,None
2,notes,Progress Notes,Past History,Organ Systems,Cardiovascular (R),Congestive Heart Failure,CHF - class II,None,None
3,notes,Progress Notes,Past History,Past History Obtain Options,Performed,None,None,None,None
4,notes,Progress Notes,Past History,Organ Systems,Cardiovascular (R),Hypertension Requiring Treatment,hypertension requiring treatment,None,None


In [16]:
# 4. Restaurar la barra original en todas las columnas del split
split_data_ph = split_data_ph.apply(lambda x: x.str.replace('TEMP_SLASH', '/', regex=False))

In [17]:
# 3. Seleccionar las columnas resultantes (usando índices 4 a 9)
cols_interes = split_data_ph[[4, 5, 6, 7, 8]].copy()
cols_interes.columns = ['Col4', 'Col5', 'Col6', 'Col7', 'Col8']

# 4. Agrupar 7, 8 y 9 con lógica ": " y omitir "None"
def unir_con_formato(row):
    # Filtramos valores que sean None, "None" (string) o vacíos
    partes = [str(val).strip() for val in [row['Col7'], row['Col8']] 
              if val and str(val).lower() != 'none' and str(val).strip() != '']
    return ": ".join(partes)

df_ph_final['Type'] = cols_interes['Col4']
df_ph_final['Subtype'] = cols_interes['Col5']
df_ph_final['PastHistory'] = cols_interes['Col6']
df_ph_final['MoreInfo'] = cols_interes.apply(unir_con_formato, axis=1)

df_ph_final.head()

,Patient,Offset,Type,Subtype,PastHistory,MoreInfo
0,141168,60,Cardiovascular (R),Valve disease,AS,
1,141168,114,Renal (R),Renal Failure,renal failure- not currently dialyzed,
2,141168,60,Cardiovascular (R),Congestive Heart Failure,CHF - class II,
3,141168,114,Performed,None,None,
4,141168,60,Cardiovascular (R),Hypertension Requiring Treatment,hypertension requiring treatment,


In [18]:
# Reemplazar los valores nulos en la columna 'Subtype'
df_ph_final['Subtype'] = df_ph_final['Subtype'].fillna('Unknown subtype')

# Reemplazar los valores nulos en la columna 'Subtype'
df_ph_final['PastHistory'] = df_ph_final['PastHistory'].fillna('Unknown PastHistory')

In [19]:
# 5. Definir la ruta completa del archivo
ruta_final_ph = os.path.join(carpeta_destino, 'PastHist_AdmFact.csv')

# 6. Guardar con separador de punto y coma
df_ph_final.to_csv(ruta_final_ph, sep=';', index=False)

print(f"¡Éxito! El archivo se ha guardado en: {ruta_final_ph}")

¡Éxito! El archivo se ha guardado en: ../data/CSV_filtrados\PastHist_AdmFact.csv


## 4. Tabla PhyEx_AdmFact
Transformación del CSV de la tabla PhysicalExam, con la misma lógica de excepciones y `split` de la columna `physicalexampath` que en PastHistory, más una limpieza adicional para evitar que el campo `MoreInfo` repita información ya presente en `Result`.

In [20]:
# 2. Leer el archivo original
df_physical = pd.read_csv('../data/csv_origen/physicalExam.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df_physical.head()

,physicalexamid,patientunitstayid,physicalexamoffset,physicalexampath,physicalexamvalue,physicalexamtext
0,5253099,176895,9,notes/Progress Notes/Physical Exam/Physical Ex...,scored,scored
1,5253100,176895,9,notes/Progress Notes/Physical Exam/Physical Ex...,Performed - Structured,Performed - Structured
2,5253104,176895,9,notes/Progress Notes/Physical Exam/Physical Ex...,Current,68.3
3,5253105,176895,9,notes/Progress Notes/Physical Exam/Physical Ex...,Intake Total,2725
4,5253106,176895,9,notes/Progress Notes/Physical Exam/Physical Ex...,Output Total,1360


In [21]:
# 2. Definir excepciones y protegerlas
# Reemplazamos las barras en las cadenas específicas por un marcador temporal (ej. "TEMP_SLASH")
excepciones_physical = ['Ears/Nose/Mouth/Throat', 'cold/dusky']
path_protegido = df_physical['physicalexampath'].copy()

for exc in excepciones_physical:
    # Usamos un marcador que no interfiera con el split
    path_protegido = path_protegido.str.replace(exc, exc.replace('/', 'TEMP_SLASH'), regex=False)

# 2. Split y restauración de la barra
split_data_physical = path_protegido.str.split('/', expand=True)
split_data_physical = split_data_physical.apply(lambda x: x.str.replace('TEMP_SLASH', '/', regex=False))

split_data_physical.head()

,0,1,2,3,4,5,6,7,8,9,10
0,notes,Progress Notes,Physical Exam,Physical Exam,Neurologic,GCS,Score,scored,None,None,None
1,notes,Progress Notes,Physical Exam,Physical Exam Obtain Options,Performed - Structured,None,None,None,None,None,None
2,notes,Progress Notes,Physical Exam,Physical Exam,Constitutional,Weight and I&O,Weight (kg),Current,None,None,None
3,notes,Progress Notes,Physical Exam,Physical Exam,Constitutional,Weight and I&O,I&&O (ml),Intake Total,None,None,None
4,notes,Progress Notes,Physical Exam,Physical Exam,Constitutional,Weight and I&O,I&&O (ml),Output Total,None,None,None


In [22]:
# 3. Seleccionar y copiar las columnas necesarias
df_physical_final_output = pd.DataFrame()
df_physical_final_output['Patient'] = df_physical['patientunitstayid']
df_physical_final_output['Offset'] = df_physical['physicalexamoffset']
df_physical_final_output['Result'] = df_physical['physicalexamtext']

df_physical_final_output['Type'] = split_data_physical[4]
df_physical_final_output['Subtype'] = split_data_physical[5]
df_physical_final_output['PhysicalExam'] = split_data_physical[6]

df_physical_final_output.head()

,Patient,Offset,Result,Type,Subtype,PhysicalExam
0,176895,9,scored,Neurologic,GCS,Score
1,176895,9,Performed - Structured,Performed - Structured,None,None
2,176895,9,68.3,Constitutional,Weight and I&O,Weight (kg)
3,176895,9,2725,Constitutional,Weight and I&O,I&&O (ml)
4,176895,9,1360,Constitutional,Weight and I&O,I&&O (ml)


In [23]:
# 1. Seleccionamos las columnas resultantes del split (índices 4 al 11)
cols_interes = split_data_physical[[7, 8, 9, 10]].copy()
cols_interes.columns = ['Col7', 'Col8', 'Col9', 'Col10']

# 2. Función con lógica de agrupación y desduplicación contra 'Result'
def unir_physical_info(row):
    # Seleccionamos los valores de las columnas 7 a 11 que no sean nulos o "None"
    lista_valores = [str(val).strip() for val in [row['Col7'], row['Col8'], row['Col9'], row['Col10']] 
                     if val and str(val).lower() != 'none' and str(val).strip() != '']
    
    if not lista_valores:
        return ""

    # Lógica de desduplicación:
    # Si el último valor de la lista es igual al valor de 'Result', lo eliminamos
    valor_result = str(row['Result']).strip()
    if lista_valores[-1] == valor_result:
        lista_valores.pop() # Quitamos el último elemento
        
    return ": ".join(lista_valores)

# 3. Construir el DataFrame final integrando la lógica anterior
# Usamos join o concat para tener acceso a la columna 'Result' dentro de la función apply
temp_df = pd.concat([df_physical_final_output, cols_interes], axis=1)

df_physical_final_output['MoreInfo'] = temp_df.apply(unir_physical_info, axis=1)

In [24]:
# Definimos los mapeos de limpieza: {Valor en Result: Cadena a eliminar en MoreInfo}
clean_map = {
    'crescendo/decrescendo': ': crescendo: decrescendo',
    'calm/appropriate': ': calm: appropriate'
}

def logic_cleaner(row):
    result_val = row['Result']
    more_info_val = str(row['MoreInfo'])
    
    # Si el valor de Result está en nuestro diccionario y la cadena existe en MoreInfo
    if result_val in clean_map:
        string_to_remove = clean_map[result_val]
        if string_to_remove in more_info_val:
            return more_info_val.replace(string_to_remove, '')
            
    return row['MoreInfo']

# Aplicamos la función al DataFrame
df_physical_final_output['MoreInfo'] = df_physical_final_output.apply(logic_cleaner, axis=1)

In [25]:
df_physical_final_output.head()

,Patient,Offset,Result,Type,Subtype,PhysicalExam,MoreInfo
0,176895,9,scored,Neurologic,GCS,Score,
1,176895,9,Performed - Structured,Performed - Structured,None,None,
2,176895,9,68.3,Constitutional,Weight and I&O,Weight (kg),Current
3,176895,9,2725,Constitutional,Weight and I&O,I&&O (ml),Intake Total
4,176895,9,1360,Constitutional,Weight and I&O,I&&O (ml),Output Total


In [26]:
# Reemplazar los valores nulos en la columna 'Subtype'
df_physical_final_output['Subtype'] = df_physical_final_output['Subtype'].fillna('Unknown subtype')

# Reemplazar los valores nulos en la columna 'Subtype'
df_physical_final_output['PhysicalExam'] = df_physical_final_output['PhysicalExam'].fillna('Unknown PhysicalExam')

In [27]:
# 5. Definir la ruta completa del archivo
ruta_final_physical = os.path.join(carpeta_destino, 'PhyEx_AdmFact.csv')

# 6. Guardar con separador de punto y coma
df_physical_final_output.to_csv(ruta_final_physical, sep=';', index=False)

print(f"¡Éxito! El archivo se ha guardado en: {ruta_final_physical}")

¡Éxito! El archivo se ha guardado en: ../data/CSV_filtrados\PhyEx_AdmFact.csv


## 5. Tabla Service
La tabla `Service` no existe como tal en los CSV de origen: se construye a partir de los valores únicos que aparecen como primer nivel en las columnas concatenadas de `diagnosis.csv` y `treatment.csv`.

### 5.1. Servicios desde Diagnosis

In [28]:
# 1. Leer el archivo original
df_diagnosis = pd.read_csv('../data/csv_origen/diagnosis.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df_diagnosis.head()

,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,4222318,141168,False,72,cardiovascular|chest pain / ASHD|coronary arte...,"414.00, I25.10",Other
1,3370568,141168,True,118,cardiovascular|ventricular disorders|cardiomyo...,NaN,Other
2,4160941,141168,False,72,pulmonary|disorders of the airways|COPD,"491.20, J44.9",Other
3,4103261,141168,True,118,pulmonary|disorders of the airways|COPD,"491.20, J44.9",Other
4,3545241,141168,True,118,cardiovascular|ventricular disorders|congestiv...,"428.0, I50.9",Other


In [29]:
# 2. Aplicar el split por el separador '|'
split_diag = df_diagnosis['diagnosisstring'].str.split('|', expand=True)

split_diag.head()

,0,1,2,3,4,5
0,cardiovascular,chest pain / ASHD,coronary artery disease,known,None,None
1,cardiovascular,ventricular disorders,cardiomyopathy,None,None,None
2,pulmonary,disorders of the airways,COPD,None,None,None
3,pulmonary,disorders of the airways,COPD,None,None,None
4,cardiovascular,ventricular disorders,congestive heart failure,None,None,None


In [30]:
# 3. Extraer valores únicos y limpiar
# Tomamos la primera columna, eliminamos duplicados y valores nulos
services_unicos_diagnosis = split_diag[0].dropna().unique()

# 4. Crear el DataFrame de salida
df_service_diagnosis = pd.DataFrame(services_unicos_diagnosis, columns=['Service'])

# 5. Ordenar alfabéticamente
df_service_diagnosis = df_service_diagnosis.sort_values(by='Service').reset_index(drop=True)

df_service_diagnosis

,Service
0,burns/trauma
1,cardiovascular
2,endocrine
3,gastrointestinal
4,general
5,genitourinary
6,hematology
7,infectious diseases
8,musculoskeletal
9,neurologic


### 5.2. Servicios desde Treatment

In [31]:
# 1. Leer el archivo original
df_treatment = pd.read_csv('../data/csv_origen/treatment.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df_treatment.head()

,treatmentid,patientunitstayid,treatmentoffset,treatmentstring,activeupondischarge
0,8399138,242040,198,cardiovascular|hypertension|angiotensin II rec...,False
1,8626134,242040,198,cardiovascular|myocardial ischemia / infarctio...,False
2,8517569,242040,198,infectious diseases|medications|therapeutic an...,False
3,9597686,242040,616,cardiovascular|non-operative procedures|diagno...,False
4,9334096,242040,618,infectious diseases|medications|therapeutic an...,True


In [32]:
# 2. Aplicar el split por el separador '|'
split_treatment = df_treatment['treatmentstring'].str.split('|', expand=True)

split_treatment.head()

,0,1,2,3,4,5
0,cardiovascular,hypertension,angiotensin II receptor blocker (ARB),losartan,None,None
1,cardiovascular,myocardial ischemia / infarction,antihyperlipidemic agent,HMG-CoA reductase inhibitor,atorvastatin,None
2,infectious diseases,medications,therapeutic antibacterials,macrolide,azithromycin,None
3,cardiovascular,non-operative procedures,diagnostic ultrasound of heart,transthoracic echocardiography,None,None
4,infectious diseases,medications,therapeutic antibacterials,vancomycin,None,None


In [33]:
# 3. Extraer valores únicos y limpiar
# Tomamos la primera columna, eliminamos duplicados y valores nulos
services_unicos_treatment = split_treatment[0].dropna().unique()

# 4. Crear el DataFrame de salida
df_service_treatment = pd.DataFrame(services_unicos_treatment, columns=['Service'])

# 5. Ordenar alfabéticamente
df_service_treatment = df_service_treatment.sort_values(by='Service').reset_index(drop=True)

df_service_treatment

,Service
0,burns/trauma
1,cardiovascular
2,endocrine
3,gastrointestinal
4,general
5,hematology
6,infectious diseases
7,neurologic
8,oncology
9,pulmonary


### 5.3. Unión de Service
Combinamos los servicios detectados en ambas fuentes, eliminamos duplicados y añadimos un valor `unknown` para los casos sin servicio identificado.

In [34]:
# Concatenamos ambos DataFrames
df_service_combined = pd.concat([df_service_diagnosis, df_service_treatment], ignore_index=True)

# 2. Obtenemos los valores únicos de la unión
# drop_duplicates() se encarga de dejar solo una instancia de cada servicio
df_service = df_service_combined.drop_duplicates().reset_index(drop=True)

# 3. Ordenar alfabéticamente para mayor orden en la tabla maestra
df_service = df_service.sort_values(by='Service').reset_index(drop=True)

df_service

,Service
0,burns/trauma
1,cardiovascular
2,endocrine
3,gastrointestinal
4,general
5,genitourinary
6,hematology
7,infectious diseases
8,musculoskeletal
9,neurologic


In [35]:
# 4. Añadimos el valor 'unknown' al final
# Creamos un pequeño DataFrame con el valor nuevo y lo añadimos
nuevo_servicio = pd.DataFrame({'Service': ['unknown']})
df_service = pd.concat([df_service, nuevo_servicio], ignore_index=True)

In [36]:
print("Valores únicos de Service:")
print(df_service)

Valores únicos de Service:
                  Service
0            burns/trauma
1          cardiovascular
2               endocrine
3        gastrointestinal
4                 general
5           genitourinary
6              hematology
7     infectious diseases
8         musculoskeletal
9              neurologic
10  obstetrics/gynecology
11               oncology
12              pulmonary
13                  renal
14                surgery
15             toxicology
16             transplant
17                unknown


In [37]:
# 5. Definir la ruta completa del archivo
ruta_final_service = os.path.join(carpeta_destino, 'Service.csv')

# 6. Guardar con separador de punto y coma
df_service.to_csv(ruta_final_service, sep=';', index=False)

print(f"¡Éxito! El archivo se ha guardado en: {ruta_final_service}")

¡Éxito! El archivo se ha guardado en: ../data/CSV_filtrados\Service.csv


## 6. Tabla AdmissionDiagnosis-Service
Relaciona cada diagnóstico de admisión (`admissionDx.csv`) con su servicio correspondiente, filtrando por el nivel `All Diagnosis` de la columna `admitdxpath`.

In [38]:
# 1. Leer el archivo original
df_admission = pd.read_csv('../data/csv_origen/admissionDx.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df_admission.head()

,admissiondxid,patientunitstayid,admitdxenteredoffset,admitdxpath,admitdxname,admitdxtext
0,8023778,2900217,62,admission diagnosis|Operative Organ Systems|Or...,Cardiovascular,Cardiovascular
1,8023777,2900217,62,admission diagnosis|Was the patient admitted f...,Yes,Yes
2,8023779,2900217,62,admission diagnosis|All Diagnosis|Operative|Di...,Aortic and Mitral valve replacement,Aortic and Mitral valve replacement
3,7929318,2900240,53,admission diagnosis|Non-operative Organ System...,Gastrointestinal,Gastrointestinal
4,7929317,2900240,53,admission diagnosis|Was the patient admitted f...,No,No


In [39]:
# 2. Dividir 'admitdxpath'
# Usamos expand=True para obtener un DataFrame de columnas
split_data_admission = df_admission['admitdxpath'].str.split('|', expand=True)

split_data_admission.head()

,0,1,2,3,4,5
0,admission diagnosis,Operative Organ Systems,Organ System,Cardiovascular,None,None
1,admission diagnosis,Was the patient admitted from the O.R. or went...,Yes,None,None,None
2,admission diagnosis,All Diagnosis,Operative,Diagnosis,Cardiovascular,Aortic and Mitral valve replacement
3,admission diagnosis,Non-operative Organ Systems,Organ System,Gastrointestinal,None,None
4,admission diagnosis,Was the patient admitted from the O.R. or went...,No,None,None,None


In [40]:
# 3. Filtrar por All Diagnosis para obtener únicamente los registros de los diagnósticos
df_filtrado_admission = split_data_admission[split_data_admission.iloc[:, 1] == 'All Diagnosis']

df_filtrado_admission.head()

,0,1,2,3,4,5
2,admission diagnosis,All Diagnosis,Operative,Diagnosis,Cardiovascular,Aortic and Mitral valve replacement
5,admission diagnosis,All Diagnosis,Non-operative,Diagnosis,Gastrointestinal,"Bleeding, GI-location unknown"
6,admission diagnosis,All Diagnosis,Non-operative,Diagnosis,Cardiovascular,"Infarction, acute myocardial (MI)"
13,admission diagnosis,All Diagnosis,Non-operative,Diagnosis,Genitourinary,"Renal failure, acute"
14,admission diagnosis,All Diagnosis,Non-operative,Diagnosis,Respiratory,Emphysema/bronchitis


In [41]:
# 4. Seleccionar las columnas 4 y 5 y limpiar nulos
cols_interes = df_filtrado_admission[[4, 5]].dropna()

# 5. Crear el DataFrame definitivo con las DOS columnas que me pides
df_final_admission = pd.DataFrame()
df_final_admission['AdmissionDiagnosis'] = cols_interes[5].astype(str).str.strip(' "')
df_final_admission['Service'] = cols_interes[4].astype(str).str.strip(' "').str.lower()

# 6. Eliminar duplicados basándonos en la combinación de ambas columnas
df_final_admission = df_final_admission.drop_duplicates()

# 7. Ordenar alfabéticamente por la primera columna (AdmissionDiagnosis)
df_final_admission = df_final_admission.sort_values(by=['AdmissionDiagnosis', 'Service']).reset_index(drop=True)

# 8. Insertar la primera fila 'Unknown' y 'unknown'
fila_unknown = pd.DataFrame({'AdmissionDiagnosis': ['Unknown'], 'Service': ['unknown']})
df_final_admission = pd.concat([fila_unknown, df_final_admission], ignore_index=True)

In [42]:
df_final_admission

,AdmissionDiagnosis,Service
0,Unknown,unknown
1,"ARDS-adult respiratory distress syndrome, non-...",respiratory
2,Abdomen only trauma,trauma
3,Abdomen/extremity trauma,trauma
4,Abdomen/face trauma,trauma
...,...,...
388,Vena cava filter insertion,cardiovascular
389,Ventricular Septal Defect (VSD) Repair,cardiovascular
390,Ventriculostomy,neurology
391,Weaning from mechanical ventilation (transfer ...,respiratory


In [43]:
# 8. Definir la ruta completa del archivo
ruta_final_admission = os.path.join(carpeta_destino, 'AdmissionDiagnosis-Service.csv')

# 9. Guardar con separador de punto y coma
df_final_admission.to_csv(ruta_final_admission, sep=':', index=False)

print(f"¡Éxito! El archivo se ha guardado en: {ruta_final_admission}")

¡Éxito! El archivo se ha guardado en: ../data/CSV_filtrados\AdmissionDiagnosis-Service.csv


### 6.1. Preservación de Compatibilidad Histórica
Para no perder las asignaciones Diagnóstico → Servicio ya validadas en el TFG anterior, fusionamos el resultado de este análisis con el fichero histórico: si un diagnóstico ya tenía un servicio asignado previamente, se respeta esa asignación; si es un diagnóstico nuevo, se usa el resultado de este cuaderno.

In [44]:
# 1. Cargar el CSV que corrigió Gemini
# Nota: Como el separador es ':', especificamos sep=':'
df_gemini = pd.read_csv('../data/csv_origen/AdmissionDiagnosis-Service_reassigned.csv', sep=':')

# 2. Cargar el CSV del TFG anterior (el histórico que debemos respetar)
df_tfg_anterior = pd.read_csv('../data/csv_origen/AdmissionDiagnosis-Service_anterior.csv', sep=':')

# Aseguramos que la columna Service del TFG anterior también esté en minúsculas
df_tfg_anterior['Service'] = df_tfg_anterior['Service'].astype(str).str.strip().str.lower()
df_tfg_anterior['AdmissionDiagnosis'] = df_tfg_anterior['AdmissionDiagnosis'].astype(str).str.strip()

# 3. Hacer un "Left Join" para identificar qué diagnósticos ya existían antes
# Cruzamos tu tabla curada con la antigua basándonos SOLO en el diagnóstico
df_fusion = pd.merge(
    df_gemini, 
    df_tfg_anterior, 
    on='AdmissionDiagnosis', 
    how='left', 
    suffixes=('_gemini', '_antiguo')
)

# 4. Lógica de prioridad: Si existe el servicio antiguo, lo usamos; si no, dejamos el de Gemini
df_fusion['Service'] = df_fusion['Service_antiguo'].fillna(df_fusion['Service_gemini'])

# 5. Limpieza: Nos quedamos solo con las dos columnas definitivas
df_final = df_fusion[['AdmissionDiagnosis', 'Service']].copy()

# Eliminar duplicados residuales si los hubiera y ordenar alfabéticamente
df_final = df_final.drop_duplicates().sort_values(by=['AdmissionDiagnosis']).reset_index(drop=True)

# 6. Volver a forzar la fila 'Unknown:unknown' en la primera posición
df_final = df_final[df_final['AdmissionDiagnosis'] != 'Unknown'] # Por si acaso se movió
fila_unknown = pd.DataFrame({'AdmissionDiagnosis': ['Unknown'], 'Service': ['unknown']})
df_final = pd.concat([fila_unknown, df_final], ignore_index=True)

# 7. Guardar el archivo final definitivo con separador ':'
ruta_final_admission = os.path.join(carpeta_destino, 'AdmissionDiagnosis-Service.csv')
df_final.to_csv(ruta_final_admission, sep=':', index=False)

print("¡Fusión completada! Se ha respetado el histórico del TFG anterior.")

¡Fusión completada! Se ha respetado el histórico del TFG anterior.


## 7. Tabla Treat_AdmFact
Transformación de `treatment.csv`: dividimos `treatmentstring` por `|` y agrupamos los niveles de detalle en las columnas `Type`, `Subtype`, `Treatment` y `MoreInfo`.

In [45]:
# 1. Leer el archivo original
df_treat = pd.read_csv('../data/csv_origen/treatment.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df_treat.head()

,treatmentid,patientunitstayid,treatmentoffset,treatmentstring,activeupondischarge
0,8399138,242040,198,cardiovascular|hypertension|angiotensin II rec...,False
1,8626134,242040,198,cardiovascular|myocardial ischemia / infarctio...,False
2,8517569,242040,198,infectious diseases|medications|therapeutic an...,False
3,9597686,242040,616,cardiovascular|non-operative procedures|diagno...,False
4,9334096,242040,618,infectious diseases|medications|therapeutic an...,True


In [46]:
# 2. Seleccionar y copiar las columnas necesarias
columnas_base = ['patientunitstayid', 'treatmentoffset', 'activeupondischarge']
df_treat_final = df_treat[columnas_base].copy()

# Renombrar las columnas
df_treat_final.columns = ['Patient', 'Offset', 'Active']
df_treat_final.head()

,Patient,Offset,Active
0,242040,198,False
1,242040,198,False
2,242040,198,False
3,242040,616,False
4,242040,618,True


In [47]:
# 3. Dividir 'treatmentstring'
# Usamos expand=True para obtener un DataFrame de columnas
split_data_treat = df_treat['treatmentstring'].str.split('|', expand=True)

split_data_treat.head()

,0,1,2,3,4,5
0,cardiovascular,hypertension,angiotensin II receptor blocker (ARB),losartan,None,None
1,cardiovascular,myocardial ischemia / infarction,antihyperlipidemic agent,HMG-CoA reductase inhibitor,atorvastatin,None
2,infectious diseases,medications,therapeutic antibacterials,macrolide,azithromycin,None
3,cardiovascular,non-operative procedures,diagnostic ultrasound of heart,transthoracic echocardiography,None,None
4,infectious diseases,medications,therapeutic antibacterials,vancomycin,None,None


In [48]:
# 4. Seleccionar las columnas resultantes (usando índices 4 a 9)
cols_interes = split_data_treat[[3, 4, 5]].copy()
cols_interes.columns = ['Col3', 'Col4', 'Col5']

# 5. Agrupar 3, 4 y 5 con lógica "|" y omitir "None"
def unir_con_formato(row):
    # Filtramos valores que sean None, "None" (string) o vacíos
    partes = [str(val).strip() for val in [row['Col3'], row['Col4'], row['Col5']] 
              if val and str(val).lower() != 'none' and str(val).strip() != '']
    return ": ".join(partes)

df_treat_final['Type'] = split_data_treat[0]
df_treat_final['Subtype'] = split_data_treat[1]
df_treat_final['Treatment'] = split_data_treat[2]
df_treat_final['MoreInfo'] = cols_interes.apply(unir_con_formato, axis=1)

df_treat_final.head()

,Patient,Offset,Active,Type,Subtype,Treatment,MoreInfo
0,242040,198,False,cardiovascular,hypertension,angiotensin II receptor blocker (ARB),losartan
1,242040,198,False,cardiovascular,myocardial ischemia / infarction,antihyperlipidemic agent,HMG-CoA reductase inhibitor: atorvastatin
2,242040,198,False,infectious diseases,medications,therapeutic antibacterials,macrolide: azithromycin
3,242040,616,False,cardiovascular,non-operative procedures,diagnostic ultrasound of heart,transthoracic echocardiography
4,242040,618,True,infectious diseases,medications,therapeutic antibacterials,vancomycin


In [49]:
# 6. Definir la ruta completa del archivo
ruta_final_treat = os.path.join(carpeta_destino, 'Treat_AdmFact.csv')

# 7. Guardar con separador de punto y coma
df_treat_final.to_csv(ruta_final_treat, sep=';', index=False)

print(f"¡Éxito! El archivo se ha guardado en: {ruta_final_treat}")

¡Éxito! El archivo se ha guardado en: ../data/CSV_filtrados\Treat_AdmFact.csv


## 8. Tabla Diag_AdmFact
Transformación de `diagnosis.csv` siguiendo la misma lógica que Treat_AdmFact: dividimos `diagnosisstring` por `|` y agrupamos los niveles de detalle en `Type`, `Subtype`, `Diagnosis` y `MoreInfo`.

In [50]:
# 1. Leer el archivo original
df_diag = pd.read_csv('../data/csv_origen/diagnosis.csv')

# Visualizamos las primeras 5 filas para entender qué tenemos
df_diag.head()

,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,4222318,141168,False,72,cardiovascular|chest pain / ASHD|coronary arte...,"414.00, I25.10",Other
1,3370568,141168,True,118,cardiovascular|ventricular disorders|cardiomyo...,NaN,Other
2,4160941,141168,False,72,pulmonary|disorders of the airways|COPD,"491.20, J44.9",Other
3,4103261,141168,True,118,pulmonary|disorders of the airways|COPD,"491.20, J44.9",Other
4,3545241,141168,True,118,cardiovascular|ventricular disorders|congestiv...,"428.0, I50.9",Other


In [51]:
# 2. Seleccionar y copiar las columnas necesarias
columnas_base = ['patientunitstayid', 'activeupondischarge', 'diagnosisoffset']
df_diag_final = df_diag[columnas_base].copy()

# Renombrar las columnas
df_diag_final.columns = ['Patient', 'Active', 'Offset']
df_diag_final.head()


,Patient,Active,Offset
0,141168,False,72
1,141168,True,118
2,141168,False,72
3,141168,True,118
4,141168,True,118


In [52]:
# 3. Dividir 'diagnosisstring'
# Usamos expand=True para obtener un DataFrame de columnas
split_data_diag = df_diag['diagnosisstring'].str.split('|', expand=True)

split_data_diag.head()

,0,1,2,3,4,5
0,cardiovascular,chest pain / ASHD,coronary artery disease,known,None,None
1,cardiovascular,ventricular disorders,cardiomyopathy,None,None,None
2,pulmonary,disorders of the airways,COPD,None,None,None
3,pulmonary,disorders of the airways,COPD,None,None,None
4,cardiovascular,ventricular disorders,congestive heart failure,None,None,None


In [53]:
# 4. Seleccionar las columnas resultantes (usando índices 4 a 9)
cols_interes = split_data_diag[[3, 4, 5]].copy()
cols_interes.columns = ['Col3', 'Col4', 'Col5']

# 5. Agrupar 3, 4 y 5 con lógica "|" y omitir "None"
def unir_con_formato(row):
    # Filtramos valores que sean None, "None" (string) o vacíos
    partes = [str(val).strip() for val in [row['Col3'], row['Col4'], row['Col5']] 
              if val and str(val).lower() != 'none' and str(val).strip() != '']
    return "|".join(partes)

df_diag_final['Type'] = split_data_diag[0]
df_diag_final['Subtype'] = split_data_diag[1]
df_diag_final['Diagnosis'] = split_data_diag[2]
df_diag_final['MoreInfo'] = cols_interes.apply(unir_con_formato, axis=1)

df_diag_final.head()

,Patient,Active,Offset,Type,Subtype,Diagnosis,MoreInfo
0,141168,False,72,cardiovascular,chest pain / ASHD,coronary artery disease,known
1,141168,True,118,cardiovascular,ventricular disorders,cardiomyopathy,
2,141168,False,72,pulmonary,disorders of the airways,COPD,
3,141168,True,118,pulmonary,disorders of the airways,COPD,
4,141168,True,118,cardiovascular,ventricular disorders,congestive heart failure,


In [54]:
# 6. Definir la ruta completa del archivo
ruta_final_diag = os.path.join(carpeta_destino, 'Diag_AdmFact.csv')

# 7. Guardar con separador de punto y coma
df_diag_final.to_csv(ruta_final_diag, sep=';', index=False)

print(f"¡Éxito! El archivo se ha guardado en: {ruta_final_diag}")

¡Éxito! El archivo se ha guardado en: ../data/CSV_filtrados\Diag_AdmFact.csv
